In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
X, y = load_diabetes(return_X_y=True)

print(X.shape, y.shape)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size= 0.2 ,random_state=42)
print(X_train.shape)
print(X_test.shape)


(442, 10) (442,)
(353, 10)
(89, 10)


In [2]:
X_train_mashtab = (X_train - X_train.mean(axis = 0))/X_train.std(axis = 0)
X_test_mastab = (X_test - X_train.mean(axis = 0))/X_train.std(axis = 0)


In [3]:
linreg_model = LinearRegression()

linreg_fit = linreg_model.fit(X_train_mashtab,y_train)
linreg_predict = linreg_fit.predict(X_test_mastab)
print(sum(linreg_fit.coef_ ** 2))
print()
print(r2_score(y_test, linreg_predict))

5130.812554097907

0.4526027629719196


In [4]:
alpha = [0.01, 0.1, 1, 10, 100, 1000]

for i in alpha:
   model =  Ridge(i)
   model.fit(X_train_mashtab, y_train)
   print(r2_score(y_test, model.predict(X_test_mastab)))
   print(sum(model.coef_ ** 2))
   print()

0.45262512889254847
5109.907148024391

0.4528177573491715
4930.0497067690785

0.45414652070698225
3709.887434003572

0.45721095677808476
1851.406172560154

0.46052426916422706
1175.3121762275894

0.34850651661612286
252.31550780524867



## Вывод: Ridge (L2-регуляризация), датасет diabetes

Обычная линейная регрессия на масштабированных данных: R² на тесте = 0.4526, сумма квадратов весов Σw² = 5130.8.

| alpha | R² на тесте | Σw² |
|---|---|---|
| 0.01 | 0.4526 | 5109.9 |
| 0.1 | 0.4528 | 4930.0 |
| 1 | 0.4541 | 3709.9 |
| 10 | 0.4572 | 1851.4 |
| 100 | 0.4605 | 1175.3 |
| 1000 | 0.3485 | 252.3 |

- При росте alpha веса уменьшаются: Σw² падает с 5130.8 до 252.3.
- При alpha = 1000 штраф становится очень большим, и модель недообучается: веса прижаты к нулю, модель почти не использует признаки, поэтому R² падает до 0.3485.
- До alpha = 100 качество не падает, а даже немного растёт (0.4526 → 0.4605), то есть умеренная регуляризация помогает. Падение начинается между alpha = 100 и alpha = 1000.
- Эффект регуляризации небольшой (прирост R² около 0.008): в данных всего 10 признаков и сильного переобучения нет.

In [5]:

for i in [0.001, 0.01, 0.1, 1, 5, 10]:
   
   model =  Lasso(i)
   model.fit(X_train_mashtab, y_train)
   truefalse = model.coef_ == 0
   print(r2_score(y_test, model.predict(X_test_mastab)))
   print(sum(model.coef_ ** 2))
   print(sum(truefalse))
   names = np.array(load_diabetes().feature_names)
   print(names[truefalse])
   print()



0.45263772902601285
5106.070324794861
0
[]

0.4529473377023646
4887.451302071672
0
[]

0.45554139902790414
3102.0737883930474
0
[]

0.46687670944102466
1697.1682212864062
1
['s2']

0.4645267396590087
1233.338038772733
5
['age' 's1' 's2' 's4' 's6']

0.44625317114112806
975.400004984343
6
['age' 'sex' 's1' 's2' 's4' 's6']



## Вывод: Lasso (L1-регуляризация), датасет diabetes

| alpha | R² на тесте | Число нулевых весов | Обнулённые признаки |
|---|---|---|---|
| 0.001 | 0.4526 | 0 | — |
| 0.01 | 0.4529 | 0 | — |
| 0.1 | 0.4555 | 0 | — |
| 1 | 0.4669 | 1 | s2 |
| 5 | 0.4645 | 5 | age, s1, s2, s4, s6 |
| 10 | 0.4463 | 6 | age, sex, s1, s2, s4, s6 |

**2.** С ростом alpha часть весов становится равной 0. Пока alpha не дошёл до значения 1, нулевых весов не было, потом их стало больше и больше (см. таблицу).

**3.** Первым обнуляется s2. Он коррелирует с s1 (≈ 0.9), и, возможно, поэтому L1 оставляет один признак из пары: при alpha = 1 обнулён только s2. Далее обнуляется и s1, и вместе с ним age, s2, s4, s6 (alpha = 5). Затем обнуляется ещё и sex (alpha = 10).

**4.** Лучшее качество показано при alpha = 1 (R² = 0.4669): возможно, из-за удаления одного лишнего признака показатели модели чуть улучшились. При дальнейших обнулениях результаты ухудшались (0.4645 и 0.4463). Прирост небольшой, и alpha здесь выбран по тестовой выборке, поэтому правильнее проверять выбор кросс-валидацией.

**5.** Lasso это L1: он может полностью занулить вес признака, в отличие от Ridge (L2), который не зануляет веса полностью, а делает их очень маленькими.

In [6]:
alpha3 = [0.01, 0.1, 1, 10, 100, 1000]
for p in alpha3:
    print(cross_val_score(make_pipeline(StandardScaler(), Ridge(alpha=p)),X_train,y_train,cv = 5, scoring='r2').mean())


0.44929211813010134
0.4495909987239603
0.4512662709279831
0.453937415795935
0.4511019449262272
0.29271116818043136


## Вывод: выбор alpha для Ridge через кросс-валидацию (5 фолдов, R²)

| alpha | Среднее CV R² |
|---|---|
| 0.01 | 0.4493 |
| 0.1 | 0.4496 |
| 1 | 0.4513 |
| 10 | 0.4539 |
| 100 | 0.4511 |
| 1000 | 0.2927 |

- Кросс-валидация выбрала alpha = 10 (0.4539), а по тестовой выборке лучшим оказался alpha = 100 (R² = 0.4605). Результат выбора разнится, потому что тест это фиксированная единичная проверка на 89 объектах, а кросс-валидация усредняет 5 проверок на разных частях обучающей выборки. Доверять лучше кросс-валидации: чем больше проверок, тем надёжнее оценка.
- Оценки при alpha = 10 и 100 очень близки (0.4539 и 0.4511), разница меньше возможного разброса между фолдами, поэтому эти значения практически равноценны. При alpha = 1000 качество резко падает (0.2927): модель недообучается.
- Гиперпараметр нужно выбирать по кросс-валидации, а тест использовать один раз в конце, чтобы честно оценить выбранную модель.